In [1]:
import torch
from torchvision.models import efficientnet_b0
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torch.nn as nn
import torchaudio
import soundfile as sf
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path    
from tqdm.notebook import tqdm    
import numpy as np
import torchaudio.transforms as T
import math

In [2]:

taxonomy_df = pd.read_csv('../data/taxonomy.csv')
taxonomy_set = sorted(set((taxonomy_df['primary_label'].unique())))
idx2label = {}
NUM_CLASSES = len(taxonomy_set)


In [3]:
submission_df = pd.DataFrame(columns = ['row_id'] + list(taxonomy_set))

In [4]:
trained_model_path = '../models/20260415_041776249939/efficientnet_b0_best_.pth'
trained_model = efficientnet_b0(weights = None)
trained_model.classifier[1] = nn.Linear(trained_model.classifier[1].in_features, NUM_CLASSES)
trained_model.load_state_dict(torch.load(trained_model_path))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
trained_model.to(device)
print('Number of parameters of the model: ', sum(p.numel() for p in trained_model.parameters()))
print('Number of trainable parameters of the model: ', sum(p.numel() for p in trained_model.parameters() if p.requires_grad))


Number of parameters of the model:  4307302
Number of trainable parameters of the model:  4307302


In [5]:
print(device)

cuda


In [8]:

FIXED_LENGTH = 5
NUM_CHUNKS = 12 # 1 minute file are divided in 12 chunks of 5 sec
SAMPLE_RATE = 32000
N_FFT = 1024
HOP_LENGTH = 512
F_MIN = 500
F_MAX = 16000
N_MELS = 128
WINDOW_FUNCTION = torch.hamming_window # hamming the edge of the windows to 0 to reduce abrupt change
TARGET_LENGTH = FIXED_LENGTH * SAMPLE_RATE
NUM_CLASSES = len(taxonomy_set) # take taxonomy set as a baseline

transform = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    f_min=F_MIN,
    f_max=F_MAX,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    window_fn=WINDOW_FUNCTION 
)
class BirdSoundScrapeDataset(Dataset):
    def __init__(self, sc_file_paths, transform):
        self.transform = transform
        self.to_db = T.AmplitudeToDB()
        self.files = sc_file_paths
    def __len__(self):
        return len(self.files) * NUM_CHUNKS
    def __getitem__(self, idx):
        file_idx = math.floor(idx/NUM_CHUNKS)
        file_remainder = idx % NUM_CHUNKS
        waveform, _ = torchaudio.load(
            self.files[file_idx],
            frame_offset=int(file_remainder)*TARGET_LENGTH,
            num_frames=TARGET_LENGTH)
        if waveform.shape[0] > 1:                          # stereo → mono, safe-guarding in case there are 2 channels
            waveform = waveform.mean(dim=0, keepdim=True)
        if waveform.shape[1] < TARGET_LENGTH: # padding to make the same size
            padding = TARGET_LENGTH - waveform.shape[1]
            waveform = nn.functional.pad(waveform, (0, padding))
        waveform = self.transform(waveform)
        waveform = self.to_db(waveform)
        waveform = waveform.repeat(3,1,1)
        row_id = str(self.files[file_idx])[:-4] + '_'+str((file_remainder+1)*FIXED_LENGTH)+'.ogg'
        return waveform, row_id



In [9]:
sc_file_paths = list(Path('../data/train_soundscapes').rglob('*.ogg'))
sc_dataset = BirdSoundScrapeDataset(sc_file_paths, transform)
reduced_sc_dataset = Subset(sc_dataset, range(30))
test_loader = DataLoader(reduced_sc_dataset, batch_size=15, shuffle=False, num_workers=4)
for waveforms, row_ids in test_loader:
    print(waveforms.shape,' ' , row_ids)
    # print(waveforms)

torch.Size([15, 3, 128, 313])   ('../data/train_soundscapes/BC2026_Train_3201_S02_20220218_001500_5.ogg', '../data/train_soundscapes/BC2026_Train_3201_S02_20220218_001500_10.ogg', '../data/train_soundscapes/BC2026_Train_3201_S02_20220218_001500_15.ogg', '../data/train_soundscapes/BC2026_Train_3201_S02_20220218_001500_20.ogg', '../data/train_soundscapes/BC2026_Train_3201_S02_20220218_001500_25.ogg', '../data/train_soundscapes/BC2026_Train_3201_S02_20220218_001500_30.ogg', '../data/train_soundscapes/BC2026_Train_3201_S02_20220218_001500_35.ogg', '../data/train_soundscapes/BC2026_Train_3201_S02_20220218_001500_40.ogg', '../data/train_soundscapes/BC2026_Train_3201_S02_20220218_001500_45.ogg', '../data/train_soundscapes/BC2026_Train_3201_S02_20220218_001500_50.ogg', '../data/train_soundscapes/BC2026_Train_3201_S02_20220218_001500_55.ogg', '../data/train_soundscapes/BC2026_Train_3201_S02_20220218_001500_60.ogg', '../data/train_soundscapes/BC2026_Train_5861_S13_20221112_001500_5.ogg', '../dat

In [ ]:
rows = []
test_loader = DataLoader(sc_dataset, batch_size=32, shuffle=False, num_workers=4)
with torch.no_grad():
    for waveforms, row_ids in tqdm(test_loader, desc=f'Predicting...'):
        waveforms = waveforms.to(device)
        preds = trained_model(waveforms)
        probs = torch.softmax(preds, dim=1).cpu().numpy()
        for row_id, prob in zip(row_ids, probs):
            rows.append([row_id] + list(prob))
    
    submission_df = pd.DataFrame(rows, columns=['row_id'] + list(taxonomy_set))
submission_df.head(2)
submission_df.to_csv('../mlruns/sumbmission.csv', index=False)